In [ ]:
# Install dependencies
!pip install bitcoinlib base58 tqdm fastecdsa numba psutil

import hashlib
import base58
from bitcoin import privkey_to_address  # Added missing import
import random
import heapq
from tqdm import tqdm
import concurrent.futures
import os
import psutil
import multiprocessing
from multiprocessing import Manager
from fastecdsa import keys, curve
import numpy as np

# Try to import Numba CUDA for GPU support
try:
    from numba import cuda
    GPU_AVAILABLE = cuda.is_available()
except Exception as e:
    print(f"GPU not available: {e}")
    GPU_AVAILABLE = False

# Set a fixed random seed for repeatability
random.seed(42)

# Target and base (using original best key with Base58 Hamming 25)
TARGET_ADDR = "1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF"
BASE_KEY = "f77dd3a23db5e0ad8a13827e8b07d923ea413990a06217a4b03ca638371c4aa0"  # Original best (Hamming 25)

# Convert base key to integer and bits
base_key_int = int(BASE_KEY, 16)
base_bits = np.array([int(b) for b in bin(base_key_int)[2:].zfill(256)], dtype=np.uint8)

# Compute Base58 Hamming distance
def base58_hamming_distance(addr1, addr2):
    return sum(c1 != c2 for c1, c2 in zip(addr1, addr2))

start_addr = privkey_to_address(BASE_KEY)
print(f"Starting address: {start_addr}")
hamming_start = base58_hamming_distance(start_addr, TARGET_ADDR)
print(f"Starting Base58 Hamming distance to target: {hamming_start}")  # Should be 25

# Resource detection
cpu_count = os.cpu_count() or 1
available_memory = psutil.virtual_memory().available / (1024 ** 3)  # Available RAM in GB
print(f"Detected {cpu_count} CPU cores")
print(f"Available memory: {available_memory:.2f} GB")
print(f"GPU available: {GPU_AVAILABLE}")

# GPU kernel (if available)
if GPU_AVAILABLE:
    @cuda.jit
    def rng_flip_kernel(bits, flip_positions, working_bits, results):
        idx = cuda.grid(1)
        if idx < flip_positions.shape[0]:
            for i in range(256):
                working_bits[idx, i] = bits[i]
            for j in range(flip_positions.shape[1]):
                pos = flip_positions[idx, j]
                if pos != -1:
                    working_bits[idx, pos] = 1 - working_bits[idx, pos]
            key_int = 0
            for i in range(256):
                key_int = (key_int << 1) | working_bits[idx, i]
            results[idx] = key_int

# Generate keys with GPU (if available)
def generate_keys_gpu(n_flips, max_bits, used_combinations):
    positions_array = np.full((n_flips, max_bits), -1, dtype=np.int32)
    for i in range(n_flips):
        positions, used_combinations = generate_unique_flip_positions(max_bits, used_combinations)
        positions_array[i, :len(positions)] = positions

    working_bits = np.zeros((n_flips, 256), dtype=np.uint8)
    d_bits = cuda.to_device(base_bits)
    d_positions = cuda.to_device(positions_array)
    d_working_bits = cuda.to_device(working_bits)
    d_results = cuda.to_device(np.zeros(n_flips, dtype=np.uint64))

    threads = 1024
    blocks = (n_flips + threads - 1) // threads
    rng_flip_kernel[blocks, threads](d_bits, d_positions, d_working_bits, d_results)
    keys_int = d_results.copy_to_host()
    return keys_int, used_combinations

# Generate keys with CPU (fallback)
def generate_keys_cpu(n_flips, max_bits, used_combinations):
    keys_int = []
    for _ in range(n_flips):
        positions, used_combinations = generate_unique_flip_positions(max_bits, used_combinations)
        key_int = base_key_int
        for pos in positions:
            if 0 <= pos < 256:
                key_int ^= (1 << (255 - pos))
        keys_int.append(key_int)
    return keys_int, used_combinations

# Generate unique flip positions (1-5 bits, no repeats)
def generate_unique_flip_positions(max_bits=5, used_combinations=None):
    if used_combinations is None:
        used_combinations = set()
    while True:
        num_flips = random.randint(1, max_bits)
        positions = random.sample(range(256), num_flips)
        positions_tuple = tuple(sorted(positions))
        if positions_tuple not in used_combinations:
            used_combinations.add(positions_tuple)
            return positions_tuple, used_combinations

# Validate a single key (optimized with fastecdsa)
def validate_key(args):
    key_int, best_hamming = args
    key_hex = hex(key_int)[2:].zfill(64)
    try:
        priv_key = int(key_hex, 16)
        pub_key = keys.get_public_key(priv_key, curve.secp256k1)
        pub_key_bytes = bytes.fromhex(f"02{pub_key.x.to_bytes(32, 'big').hex()}" if pub_key.y % 2 == 0 else f"03{pub_key.x.to_bytes(32, 'big').hex()}")
        ripemd160 = hashlib.new('ripemd160')
        sha = hashlib.sha256(pub_key_bytes).digest()
        ripe = ripemd160.copy().update(sha).digest()
        addr = base58.b58encode_check(b'\x00' + ripe).decode('ascii')
        hamming = base58_hamming_distance(addr, TARGET_ADDR)
        if hamming < best_hamming.value:
            with best_hamming.get_lock():
                if hamming < best_hamming.value:
                    best_hamming.value = hamming
                    print(f"\nNew best found: Hamming {hamming}: {key_hex} -> {addr}", flush=True)
        return (hamming, key_hex, addr)
    except Exception as e:
        print(f"Error validating key {key_hex}: {e}", flush=True)
        # Fallback to bitcoin library if fastecdsa fails
        addr = privkey_to_address(key_hex)
        hamming = base58_hamming_distance(addr, TARGET_ADDR)
        if hamming < best_hamming.value:
            with best_hamming.get_lock():
                if hamming < best_hamming.value:
                    best_hamming.value = hamming
                    print(f"\nNew best found: Hamming {hamming}: {key_hex} -> {addr}", flush=True)
        return (hamming, key_hex, addr)

# Process a batch for a single process
def process_batch(args):
    batch_flips, max_bits, best_hamming, seen_keys, top_20, use_gpu = args
    local_results = []
    local_seen_keys = set()
    used_combinations = set()

    # Generate keys
    if use_gpu and GPU_AVAILABLE:
        keys_int, used_combinations = generate_keys_gpu(batch_flips, max_bits, used_combinations)
    else:
        keys_int, used_combinations = generate_keys_cpu(batch_flips, max_bits, used_combinations)

    # Validate keys with ThreadPoolExecutor
    with concurrent.futures.ThreadPoolExecutor() as executor:
        results = list(executor.map(validate_key, [(key_int, best_hamming) for key_int in keys_int]))
        for hamming, key_hex, addr in results:
            if hamming == 0:
                return (hamming, key_hex, addr)
            if key_hex not in seen_keys and hamming < float('inf'):
                local_seen_keys.add(key_hex)
                local_results.append((hamming, key_hex, addr))

    # Update shared seen_keys and top_20
    with seen_keys.get_lock(), top_20.get_lock():
        seen_keys.update(local_seen_keys)
        for entry in local_results:
            if len(top_20) < 20:
                heapq.heappush(top_20, entry)
            elif entry[0] < top_20[0][0]:
                heapq.heapreplace(top_20, entry)
    return None

# Main bit-flipping function
def bit_flip_cpu_rng(total_flips=10**8, max_bits=5):
    flips_done = 0

    # Use os.cpu_count() for dynamic workers
    max_workers = os.cpu_count() or 1
    batch_size = 50000 * max_workers  # Scale batch size with CPU cores
    print(f"Using {max_workers} CPU workers (Colab Pro+)", flush=True)
    print(f"Batch size: {batch_size} flips", flush=True)
    print(f"GPU acceleration: {'Enabled' if GPU_AVAILABLE else 'Disabled'}", flush=True)

    # Use multiprocessing Manager for shared state
    manager = Manager()
    best_hamming = manager.Value('i', hamming_start)
    seen_keys = manager.dict()
    top_20 = manager.list()

    with tqdm(total=total_flips, desc="Bit-Flipping (Optimized)", unit="flips") as pbar:
        while flips_done < total_flips:
            n_flips = min(batch_size, total_flips - flips_done)
            print(f"\nStarting batch: {n_flips} flips", flush=True)

            # Split batch across workers
            flips_per_worker = n_flips // max_workers
            remaining_flips = n_flips % max_workers
            batch_args = [
                (flips_per_worker + (1 if i < remaining_flips else 0), max_bits, best_hamming, seen_keys, top_20, GPU_AVAILABLE)
                for i in range(max_workers)
            ]

            # Process with ProcessPoolExecutor
            with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
                results = list(executor.map(process_batch, batch_args))
                for result in results:
                    if result and result[0] == 0:
                        hamming, key_hex, addr = result
                        print(f"\nFOUND: {key_hex} -> {addr}", flush=True)
                        with open('/content/drive/MyDrive/found_key.txt', 'w') as f:
                            f.write(f"{key_hex}\n{addr}")
                        return key_hex

            flips_done += n_flips
            pbar.update(n_flips)
            if flips_done % (batch_size * 2) == 0 or flips_done == total_flips:
                print(f"\nFlips: {flips_done}/{total_flips}", flush=True)
                print("Top 10:", flush=True)
                for hamming, key, addr in sorted(top_20)[:10]:
                    print(f"Hamming {hamming}: {addr} ({key[:16]}...)", flush=True)
                with open('/content/drive/MyDrive/top_20_bitflip_optimized.txt', 'w') as f:
                    for hamming, key, addr in sorted(top_20):
                        f.write(f"Hamming {hamming}: {key} -> {addr}\n")

    print("Done - no exact match", flush=True)
    return None

# Run the script
if __name__ == "__main__":
    bit_flip_cpu_rng(total_flips=10**8, max_bits=5)

Starting address: 17RPdVRKAHTyywX4aYMhfWdoTgeENsfguF
Starting Base58 Hamming distance to target: 25
Detected 8 CPU cores
Available memory: 49.07 GB
GPU available: False
Using 8 CPU workers (Colab Pro+)
Batch size: 400000 flips
GPU acceleration: Disabled


Bit-Flipping (Optimized):   0%|          | 0/100000000 [00:00<?, ?flips/s]

Streaming output truncated to the last 5000 lines.




Error validating key f77fd3a23db5e0ad8a13827e8b07d923aa413d90a26217a4b03ca638371c4aa0: unsupported hash type ripemd160Error validating key ff7dd3a23db5e0ad8a13827e8b07d923ea413990a06217a4b03ca638371c4aa0: unsupported hash type ripemd160Error validating key f775d3a23db5e0ad8a13827e8b07d923ea413990a06217a4b03c8638371d4aa0: unsupported hash type ripemd160Error validating key f77dd3e23db5e0ad8a13827e8b05d923ea403992a06217a4b03ca63a371c4aa0: unsupported hash type ripemd160


Error validating key f77dd3a23db5e0ad8a13827e8b17d923ea41b990a26217a4b03ca638379c4aa0: unsupported hash type ripemd160Error validating key f77dd3a23db5e0ad8a13827e8b07d923ea417990a06217a4b03ca638371c4aa0: unsupported hash type ripemd160Error validating key f77dd3a23db5e0ad8a13827e8b07d923ea413990a06217a4b03ca638371c4ea0: unsupported hash type ripemd160Error validating key f77ddba23db5e0ad8a13927e8b47d923ea413990a06217a6b03ca638771c4aa0: unsupported hash type ripemd1

Bit-Flipping (Optimized):   0%|          | 0/100000000 [00:13<?, ?flips/s]


KeyboardInterrupt: 